In [1]:
import torch
import torch.nn as nn

In [2]:
import os
os.chdir("..")

In [11]:
from src.models.image_encoder_smallcnn import ImageEncoder_SmallCNN

In [12]:
model_cnn = ImageEncoder_SmallCNN(embed_dim=128)
input_cnn = torch.randn(2, 3, 224, 224)  # should output (2, 128)  (B, embed_dim)
output_cnn = model_cnn.forward(input_cnn)
print(output_cnn.shape)

torch.Size([2, 128])


In [16]:
# lightweight or not
total_paras_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f"Num of parameters of cnn: {total_paras_cnn:,}({total_paras_cnn/1e6:.2f}M)")

Num of parameters of cnn: 487,584(0.49M)


In [14]:
from src.models.image_encoder_resnet18 import ImageEncoder_ResNet18

In [17]:
model_resnet18 = ImageEncoder_ResNet18(embed_dim=256)
input_resnet18 = torch.randn(4, 3, 224, 224)  # should output (4, 256)
output_resnet18 = model_resnet18.forward(input_resnet18)
print(output_resnet18.shape)

torch.Size([4, 256])


In [18]:
total_paras_resnet18 = sum(p.numel() for p in model_resnet18.parameters())
print(f"Num of parameters of resnet18: {total_paras_resnet18:,}({total_paras_resnet18/1e6:.2f}M)")

Num of parameters of resnet18: 11,570,496(11.57M)


In [19]:
from src.models.image_encoder_resnet50 import ImageEncoder_ResNet50

In [22]:
model_resnet50 = ImageEncoder_ResNet50(embed_dim=512)
input_resnet50 = torch.randn(10, 3, 224, 224)  # should output (10, 256)
output_resnet50 = model_resnet18.forward(input_resnet50)
print(output_resnet50.shape)

torch.Size([10, 256])


In [23]:
total_paras_resnet50 = sum(p.numel() for p in model_resnet50.parameters())
print(f"Num of parameters of resnet50: {total_paras_resnet50:,}({total_paras_resnet50/1e6:.2f}M)")

Num of parameters of resnet50: 28,753,472(28.75M)


In [24]:
from src.models.text_encoder_textcnn import TextEncoder_TextCNN

In [29]:
model_textcnn = TextEncoder_TextCNN(vocab_size=5000, embed_dim=128)
# create a fake word form
text_ids = torch.randint(0, 5000, (4, 15))  # (4, 15)
# forward
output_textcnn = model_textcnn(text_ids)
print(f"Input_textcnn:  {text_ids.shape}")   # (4, 15)
print(f"Output_textcnn: {output_textcnn.shape}")        # should be (4, 128)

Input_textcnn:  torch.Size([4, 15])
Output_textcnn: torch.Size([4, 128])


In [31]:
total_paras_textcnn = sum(p.numel() for p in model_textcnn.parameters())
# number of parameters for vocab_size=5000
print(f"Num of parameter of textcnn: {total_paras_textcnn:,} ({total_paras_textcnn/1e6:.2f}M)")

Num of parameter of textcnn: 1,031,328 (1.03M)


In [33]:
import numpy as np
import torch.nn.functional as F

In [36]:
# try to forward with text encoder and image encoder simutaneously
# temperately ignore temperature
image_enc = ImageEncoder_SmallCNN(embed_dim=128)
text_enc  = TextEncoder_TextCNN(vocab_size=5000, embed_dim=128)

# create data
images   = torch.randn(4, 3, 224, 224)      # (4, 3, 224, 224)
text_ids = torch.randint(0, 5000, (4, 15))  # (4, 15)

# forward
img_out  = image_enc(images)    # (4, 128)
text_out = text_enc(text_ids)   # (4, 128)

# L2 normalize(as in original CLIP), ready for dot product
img_out  = F.normalize(img_out,  dim=-1)
text_out = F.normalize(text_out, dim=-1)

# calculate similarity matrix
# @ means matrix multiplication: similarity_matrix[i, j] = Ii dot-product Tj
similarity_matrix = img_out @ text_out.T   # (4, 4)  # T means transpose

print(f"img_out shape:  {img_out.shape}")
print(f"text_out shape: {text_out.shape}")
print(f"similarity matrix shape:    {similarity_matrix.shape}")   # should be (4, 4)
print(f"similarity_matrix[0]: {similarity_matrix[0].detach().numpy()}")  # similarity of image0 and all the texts



img_out shape:  torch.Size([4, 128])
text_out shape: torch.Size([4, 128])
similarity matrix shape:    torch.Size([4, 4])
similarity_matrix[0]: [-0.08921599 -0.06121613 -0.05779527 -0.03359077]
